# Actor GraphMAE with the DGMAE authors' configuration
Use the DGMAE repository's Actor data loader, model, and training settings
from configs.yml. Train only the feature reconstruction objective and
plot cosine similarity on unique cross-label edges with seed 0.


In [ ]:
MODE = "configure"  # "configure", "smoke" or "full"


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

assert MODE in {"configure", "smoke", "full"}
work = Path("/tmp/dgmae-figure8-actor-author-model")
checkout = work / "source"
graphmae = work / "dgmae-official"
experiment_commit = "f28b94567451e0f6f654b54076dca04af3b31e04"
graphmae_commit = "0e770864b5283b01e2597d6f092e8a812aa5031a"

def clone(url, commit, destination):
    destination.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q", str(destination)], check=True)
    subprocess.run(["git", "-C", str(destination), "fetch", "--depth=1", url, commit], check=True)
    subprocess.run(["git", "-C", str(destination), "checkout", "--detach", commit], check=True)

if MODE != "configure":
    subprocess.run(["nvidia-smi", "-L"], check=True)
    clone("https://github.com/phud774/CS313.R11.git", experiment_commit, checkout)
    clone("https://github.com/zhengziyu77/DGMAE.git", graphmae_commit, graphmae)
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.13.7"], check=True)
    setup_dir = checkout / "experiments" / "ablation"
    subprocess.run(["bash", str(setup_dir / "setup_kaggle.sh"), str(work)], check=True)
    subprocess.run(["uv", "pip", "install", "--python", str(work / ".dgmae-venv/bin/python"),
                    "matplotlib==3.9.4", "tensorboardX==2.6.2.2"], check=True)


In [ ]:
if MODE != "configure":
    from IPython.display import Image, display

    script = checkout / "DGMAE-REIMPL/experiments/figure8/run_actor_author_config.py"
    output = Path("/kaggle/working/actor-graphmae-author-config")
    env = dict(
        os.environ, PYTHONUNBUFFERED="1", DGLBACKEND="pytorch", OMP_NUM_THREADS="4",
        DGL_DOWNLOAD_DIR=str(work / "datasets"), DGL_REPO="https://dgl-data.s3-accelerate.amazonaws.com",
    )
    stages = ("smoke", "full") if MODE == "full" else ("smoke",)
    for stage in stages:
        destination = output / stage
        cmd = [
            "bash", str(setup_dir / "run_kaggle_python.sh"), str(work), "-B", str(script),
            "--dgmae-repo", str(graphmae), "--output", str(destination),
        ]
        if stage == "smoke":
            cmd.append("--smoke")
        subprocess.run(cmd, env=env, check=True)
        display(Image(filename=str(destination / "figure8.png")))
